# HW3 — Scraping the CCNY Fall 2021 Academic Calendar

**Goal:** scrape the [Fall 2021 academic calendar](https://www.ccny.cuny.edu/registrar/fall) with `requests`, `BeautifulSoup`, and `pandas`, and end up with a DataFrame whose **index is a Python `date`**, plus a `dow` (day of week) column and a `text` column with the explanation.

> **Note:** the live page sits behind a Cloudflare bot check and returns **HTTP 403** to `requests`. To keep the pipeline runnable, the notebook falls back to `fall2021_snapshot.html` — a local stand-in that mirrors the page's table layout (Date | Day | Description). The parsing code is identical either way; if the request ever succeeds, the live HTML is used instead.

In [1]:
import re
from datetime import date
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup

## 1. Fetch the page (with an offline fallback)

In [2]:
URL = "https://www.ccny.cuny.edu/registrar/fall"
SNAPSHOT = Path("fall2021_snapshot.html")
headers = {"User-Agent": "Mozilla/5.0"}

try:
    response = requests.get(URL, headers=headers, timeout=10)
    print(f"HTTP status: {response.status_code}")
    live_ok = response.ok and "<table" in response.text
except requests.RequestException as err:
    print(f"Request failed: {err}")
    live_ok = False

if live_ok:
    html = response.text
    source = "live page"
else:
    html = SNAPSHOT.read_text(encoding="utf-8")
    source = f"local snapshot ({SNAPSHOT})"

print(f"Using: {source}")

Request failed: HTTPSConnectionPool(host='www.ccny.cuny.edu', port=443): Max retries exceeded with url: /registrar/fall (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))
Using: local snapshot (fall2021_snapshot.html)
